In [0]:
import json
import sys
from pathlib import Path

dbutils.widgets.text("repo_root", "")
supplied = dbutils.widgets.get("repo_root").strip()

REPO_ROOT = Path(supplied).resolve() if supplied else next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "config/runtime.yaml").is_file()
        and (p / "scripts/run_config.py").is_file()
    ),
    None,
)

if REPO_ROOT is None:
    raise ValueError("Set repo_root to the Workspace Git folder root")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.run_config import notebook_context

REPO_ROOT, config, runtime, controls = notebook_context(
    spark, dbutils
)

In [0]:
from scripts.layer_runner import (
    prepare_delivery,
    run_delivery_bronze,
)

job_id = dbutils.widgets.get("job_id")
job_run_id = dbutils.widgets.get("job_run_id")

trigger = dbutils.widgets.get("trigger_type").upper()

trigger = (
    "MANUAL"
    if trigger in {"MANUAL", "ONE_TIME", "RUN_JOB_TASK"}
    else "SCHEDULED"
)

delivery = prepare_delivery(
    spark,
    config,
    runtime,
    controls,
    job_id=job_id,
    job_run_id=job_run_id,
    trigger_type=trigger,
)

result = run_delivery_bronze(
    spark,
    config,
    controls,
    delivery,
)

print(
    json.dumps(
        {"delivery": delivery, "bronze": result},
        default=str,
        indent=2,
    )
)

if job_run_id:
    dbutils.jobs.taskValues.set(
        key="delivery_context",
        value=json.dumps(delivery),
    )

In [0]:
display(
    spark.table(control_config["tables"]["master"])
    .filter(F.col("run_id") == silver_run_id)
    .select(
        "control_id",
        "target_table",
        "final_status",
        "write_outcome",
        "read_count",
        "committed_count",
        "target_delta_version",
        "failure_reason",
    )
)